# 04 - Results Analysis

Load trained model results and generate paper-quality comparison figures.
Compare DDPM against MLE and Linear Inversion baselines across shot counts and state types.

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
import torch

from src.evaluation.metrics import fidelity, trace_distance, purity
from src.evaluation.plotting import (
    plot_fidelity_vs_shots,
    plot_density_matrix,
    plot_training_curves,
    plot_state_type_breakdown
)
from src.representation.cholesky import cholesky_to_dm
from src.data.states import generate_single_state
from src.data.measurements import simulate_measurements
from src.evaluation.baselines import mle_reconstruct, linear_inversion

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

## 1. Load Trained Model

Point to your trained checkpoint. Adjust the path as needed.

In [ ]:
from src.models.diffusion import DDPM

# Adjust this path to your trained checkpoint
CHECKPOINT_PATH = '../outputs/abl_stage0/checkpoints/best.pt'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

checkpoint_exists = os.path.exists(CHECKPOINT_PATH)
if checkpoint_exists:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    config = checkpoint.get('config', {})
    n_qubits = config.get('n_qubits', 2)
    d = 2**n_qubits
    cond_dim = 6**n_qubits
    
    model = DDPM(
        d=d, timesteps=config.get('diffusion_timesteps', 1000),
        cond_input_dim=cond_dim,
        base_channels=config.get('model', {}).get('base_channels', 64),
        dim_mults=tuple(config.get('model', {}).get('dim_mults', (1, 2, 4))),
        cond_dim=config.get('model', {}).get('cond_dim', 128),
        cond_dropout_prob=config.get('model', {}).get('cond_dropout_prob', 0.1),
        beta_schedule=config.get('beta_schedule', 'cosine'),
    ).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    print(f'Model loaded: n_qubits={n_qubits}, d={d}')
    print(f'Epoch: {checkpoint.get("epoch", "unknown")}')
    print(f'Best val loss: {checkpoint.get("best_val_loss", "unknown"):.6f}')
else:
    print(f'Checkpoint not found at {CHECKPOINT_PATH}')
    print('Train a model first: python experiments/train.py --n_qubits 2')
    print('Continuing with synthetic results for demonstration...')

## 2. Fidelity vs Measurement Shots

The central scientific question: Does DDPM outperform MLE at low shot counts?

In [ ]:
if checkpoint_exists:
    # Run actual evaluation
    from src.data.dataset import QSTDataset
    
    test_dataset = QSTDataset(
        n_qubits=n_qubits, n_states=50, n_shots=10000,
        regularization_eps=1e-6, seed=config.get('data', {}).get('seed', 42) + 2000,
    )
    
    shot_levels = [100, 200, 500, 1000, 2000, 5000, 10000]
    ddpm_fids = []
    ddpm_stds = []
    mle_fids = []
    mle_stds = []
    
    for n_shots in shot_levels:
        ddpm_batch = []
        mle_batch = []
        
        for i in range(min(20, len(test_dataset))):
            rho_true = test_dataset.density_matrices[i]
            meas_freqs = simulate_measurements(rho_true, n_shots=n_shots, seed=i)
            
            # DDPM
            condition = torch.from_numpy(meas_freqs.astype(np.float32)).unsqueeze(0).to(device)
            with torch.no_grad():
                x_pred = model.ddim_sample(condition, ddim_steps=100, progress=False)
            rho_ddpm = cholesky_to_dm(x_pred.cpu().numpy()[0])
            ddpm_batch.append(fidelity(rho_true, rho_ddpm))
            
            # MLE
            rho_mle = mle_reconstruct(meas_freqs, n_shots=n_shots, max_iter=5000)
            mle_batch.append(fidelity(rho_true, rho_mle))
        
        ddpm_fids.append(np.mean(ddpm_batch))
        ddpm_stds.append(np.std(ddpm_batch))
        mle_fids.append(np.mean(mle_batch))
        mle_stds.append(np.std(mle_batch))
        print(f'Shots={n_shots:5d}: DDPM={ddpm_fids[-1]:.4f}, MLE={mle_fids[-1]:.4f}')
    
    plot_fidelity_vs_shots(
        shot_levels, np.array(ddpm_fids), np.array(ddpm_stds),
        np.array(mle_fids), np.array(mle_stds),
        title=f'QST Fidelity Comparison (n={n_qubits})',
        save_path='fidelity_vs_shots.png'
    )
else:
    # Demo with placeholder data
    shot_levels = [100, 200, 500, 1000, 2000, 5000, 10000]
    ddpm_demo = np.array([0.65, 0.73, 0.82, 0.88, 0.92, 0.95, 0.97])
    mle_demo = np.array([0.55, 0.68, 0.78, 0.86, 0.91, 0.95, 0.97])
    
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.errorbar(shot_levels, ddpm_demo, yerr=0.02, marker='o', label='DDPM (expected)', color='#2196F3')
    ax.errorbar(shot_levels, mle_demo, yerr=0.02, marker='s', label='MLE (expected)', color='#FF5722')
    ax.set_xscale('log')
    ax.set_xlabel('Number of Measurement Shots')
    ax.set_ylabel('Fidelity')
    ax.set_title('Expected: Fidelity vs Shot Count (2 qubits)')
    ax.legend()
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    print('Note: Using placeholder data. Train a model for actual results.')

## 3. Training Curves

In [ ]:
if checkpoint_exists and 'metrics_history' in checkpoint:
    plot_training_curves(
        checkpoint['metrics_history'],
        save_path='training_curves.png'
    )
else:
    # Demo training curves
    fig, ax = plt.subplots(figsize=(8, 4))
    epochs = np.arange(1, 301)
    # Typical DDPM training: loss decreases smoothly
    train_loss = 1.2 * np.exp(-epochs / 80) + 0.05
    val_loss = 1.3 * np.exp(-epochs / 90) + 0.08
    ax.plot(epochs, train_loss, label='Train Loss', alpha=0.7)
    ax.plot(epochs[::10], val_loss[::10], 'o', label='Val Loss', markersize=3)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('DDPM Loss')
    ax.set_title('Typical DDPM Training Curves')
    ax.legend()
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 4. Performance by State Type

Which types of states does the diffusion model reconstruct best?

In [ ]:
state_types = ['pure_haar', 'mixed_hs', 'mixed_ginibre', 'thermal', 'product']
n_qubits_test = 2

if checkpoint_exists:
    ddpm_by_type = []
    mle_by_type = []
    
    for stype in state_types:
        fids_ddpm = []
        fids_mle = []
        for i in range(10):
            rho = generate_single_state(n_qubits_test, stype, seed=i)
            freqs = simulate_measurements(rho, n_shots=5000, seed=i)
            
            cond = torch.from_numpy(freqs.astype(np.float32)).unsqueeze(0).to(device)
            with torch.no_grad():
                x_pred = model.ddim_sample(cond, ddim_steps=100, progress=False)
            rho_ddpm = cholesky_to_dm(x_pred.cpu().numpy()[0])
            fids_ddpm.append(fidelity(rho, rho_ddpm))
            
            rho_mle = mle_reconstruct(freqs, n_shots=5000, max_iter=5000)
            fids_mle.append(fidelity(rho, rho_mle))
        
        ddpm_by_type.append(np.mean(fids_ddpm))
        mle_by_type.append(np.mean(fids_mle))
    
    plot_state_type_breakdown(state_types, np.array(ddpm_by_type), np.array(mle_by_type))
else:
    print('Train a model to see actual state-type breakdown.')
    print('Expected: DDPM performs best on pure and product states (structured),')
    print('          MLE handles highly mixed states well (less structure to learn).')

## 5. Single Reconstruction Example

In [ ]:
# Reconstruct a single state and compare DDPM vs MLE visually
n_qubits_vis = 2
d_vis = 2**n_qubits_vis

# Generate a Bell state (maximally entangled)
bell = np.zeros((d_vis, d_vis), dtype=np.complex128)
bell[0, 0] = bell[0, 3] = bell[3, 0] = bell[3, 3] = 0.5

meas_freqs = simulate_measurements(bell, n_shots=1000, seed=42)

if checkpoint_exists:
    cond = torch.from_numpy(meas_freqs.astype(np.float32)).unsqueeze(0).to(device)
    with torch.no_grad():
        x_pred = model.ddim_sample(cond, ddim_steps=100, progress=False)
    rho_ddpm = cholesky_to_dm(x_pred.cpu().numpy()[0])
else:
    # Placeholder: DDPM would reconstruct this
    rho_ddpm = bell + 0.05 * np.random.randn(d_vis, d_vis)
    rho_ddpm = (rho_ddpm + rho_ddpm.T.conj()) / 2
    eigvals = np.linalg.eigvalsh(rho_ddpm)
    eigvals = np.clip(eigvals, 0, None)
    eigvals = eigvals / np.sum(eigvals)
    eigvecs = np.linalg.eigh(rho_ddpm)[1]
    rho_ddpm = eigvecs @ np.diag(eigvals) @ eigvecs.T.conj()

rho_mle = mle_reconstruct(meas_freqs, n_shots=1000, max_iter=5000)

plot_density_matrix(
    bell, rho_ddpm,
    title=f'Bell State Reconstruction (1000 shots)\nDDPM Fidelity={fidelity(bell, rho_ddpm):.4f}, MLE Fidelity={fidelity(bell, rho_mle):.4f}',
    save_path='reconstruction_example.png'
)

## Key Research Questions

After running full experiments, answer these questions:

1. **Low-shot regime**: At what shot count does DDPM begin to outperform MLE?
2. **State type dependence**: Is DDPM's advantage larger for structured states (pure, product)?
3. **Scaling**: How does the DDPM-MLE gap change with qubit count (1, 2, 3)?
4. **Inference speed**: DDPM (DDIM 100 steps) vs MLE (L-BFGS 5000 iters) — which is faster?
5. **Conditioning strength**: Does the model actually use measurement data, or memorize training states?

The most scientifically interesting result would be DDPM > MLE in the low-shot regime
(N < 10^3 for 3 qubits), demonstrating the value of learned priors for QST.